# Collect: DoD contract awards from USAspending

Pulls Department of Defense contract awards for nine companies, one fiscal year at a time, from the USAspending API (`spending_by_award`). Every response is saved to `data/raw/` before anything else happens to it.

In [1]:
import json
import os
import time

import requests
import pandas as pd

# Every request says who is asking
HEADERS = {"User-Agent": "WebDataScience/1.0 (INFO 4617; brkr6998@colorado.edu)"}

AWARD_URL = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
COUNT_URL = "https://api.usaspending.gov/api/v2/search/spending_by_award_count/"

FIELDS = ["Award ID", "Recipient Name", "Award Amount", "Start Date", "End Date",
          "Awarding Sub Agency", "Description", "NAICS", "PSC",
          "Contract Award Type", "recipient_id"]

# recipient_search_text takes a list, so a company can have several legal names
COMPANIES = {
    "Lockheed Martin": ["Lockheed Martin"],
    "RTX": ["Raytheon", "RTX Corporation", "Pratt & Whitney", "Collins Aerospace", "Rockwell Collins"],
    "Boeing": ["Boeing"],
    "General Dynamics": ["General Dynamics", "Electric Boat", "Bath Iron Works"],
    "Northrop Grumman": ["Northrop Grumman"],
    "Palantir": ["Palantir"],
    "Anduril": ["Anduril"],
    "Shield AI": ["Shield AI"],
    "Scale AI": ["Scale AI"],
}

FISCAL_YEARS = range(2020, 2027)

# 10 pages of 100, sorted by dollars, keeps each company's 1,000 largest awards per year
MAX_PAGES = 10

os.makedirs("data/raw/pages", exist_ok=True)

In [2]:
def award_filters(company, fiscal_year):
    """The search filters for one company in one fiscal year (October through September)."""
    return {
        "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Defense"}],
        "time_period": [{"start_date": f"{fiscal_year - 1}-10-01", "end_date": f"{fiscal_year}-09-30"}],
        "award_type_codes": ["A", "B", "C", "D"],
        "recipient_search_text": COMPANIES[company],
    }


def get_page(company, fiscal_year, page):
    """Return one page of awards, using the saved copy if this page was already pulled."""
    path = f"data/raw/pages/{company.replace(' ', '_')}_FY{fiscal_year}_p{page}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)

    body = {
        "filters": award_filters(company, fiscal_year),
        "fields": FIELDS,
        "limit": 100,
        "page": page,
        "sort": "Award Amount",
        "order": "desc",
    }
    response = requests.post(AWARD_URL, json=body, headers=HEADERS, timeout=120)
    response.raise_for_status()
    data = response.json()

    with open(path, "w") as f:
        json.dump(data, f)
    time.sleep(1)
    return data


def get_awards(company, fiscal_year):
    """Page through one company's awards for one fiscal year, up to MAX_PAGES pages."""
    rows = []
    for page in range(1, MAX_PAGES + 1):
        data = get_page(company, fiscal_year, page)
        rows.extend(data["results"])
        if not data["page_metadata"]["hasNext"]:
            break
    return rows

Test on one company and year before pulling everything.

In [3]:
test_rows = get_awards("Anduril", 2024)
print(len(test_rows), "awards")
print(json.dumps(test_rows[0], indent=2))

94 awards
{
  "internal_id": 306338225,
  "Award ID": "H9240224F0057",
  "Recipient Name": "ANDURIL INDUSTRIES, INC.",
  "Award Amount": 249978466.39,
  "Start Date": "2024-08-31",
  "End Date": "2027-07-31",
  "Awarding Sub Agency": "U.S. Special Operations Command",
  "Description": "ROADRUNNER, ANVIL AND QUASAR PACKAGES",
  "NAICS": {
    "code": "334511",
    "description": "SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERONAUTICAL, AND NAUTICAL SYSTEM AND INSTRUMENT MANUFACTURING"
  },
  "PSC": {
    "code": "5340",
    "description": "HARDWARE, COMMERCIAL"
  },
  "Contract Award Type": "DELIVERY ORDER",
  "recipient_id": "1869c03a-77bb-b66c-469b-988719bbec4c-C",
  "generated_internal_id": "CONT_AWD_H9240224F0057_9700_H9240222D0001_9700"
}
